# Seed variance table (the ruler)

RMSE / CSI@0.05 / CSI@0.30 for each seed of the best-sweep config (seeds 666, 1, 200, 450, 800).

The mean ± spread of these values is the reference interval to judge every later change.

In [1]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # force CPU
os.environ['OMP_NUM_THREADS'] = '1'        # single-thread: this machine's torch crashes otherwise
os.environ['MKL_NUM_THREADS'] = '1'

# notebook lives in utils/, repo root is one level up
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'utils' else os.getcwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
import wandb
import numpy as np
import pandas as pd

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset
from utils.visualization import PlotRollout
from training.train import LightningTrainer

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')

print(REPO_ROOT)

/p/11210554-dtc-hydrology-next/marrocol/mSWE-GNN_marg


## Config and test dataset (loaded once, shared by all checkpoints)

In [2]:
CONFIG = 'config_best_sweep.yaml'

cfg = read_config(CONFIG)
wandb.init(mode='disabled', project='mswe-gnn', config=cfg)
fix_dict_in_config(wandb)
config = wandb.config
device = torch.device('cpu')

_, _, test_dataset, scalers = create_model_dataset(
    scalers=config.scalers, device=device,
    **config.dataset_parameters,
    **config.selected_node_features,
    **config.selected_edge_features
)
temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(
    config, config.temporal_dataset_parameters
)
temporal_test_dataset = to_temporal_dataset(
    test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters
)
num_node_features = temporal_test_dataset[0].x.size(-1)
num_edge_features = temporal_test_dataset[0].edge_attr.size(-1)
print('WD shape:', test_dataset[0].WD.shape)

The validation dataset you are using is the training one. Careful!


WD shape: torch.Size([30079, 119])


## Checkpoints

Each run saved two checkpoints: `<name>.h5` (best val_loss epoch) and `<name>_bestCSI.h5` (best val_CSI_005 epoch).

Fix the seed-666 paths if the original best-sweep run used a different output name.

In [3]:
CHECKPOINTS = {
    666: {'best_val_loss': 'results/best_sweep_new_gt.h5',
          'best_CSI':      'results/best_sweep_new_gt_bestCSI.h5'},
    1:   {'best_val_loss': 'results/best_sweep_seed1_v2.h5',
          'best_CSI':      'results/best_sweep_seed1_v2_bestCSI.h5'},
    200: {'best_val_loss': 'results/best_sweep_seed200_v2.h5',
          'best_CSI':      'results/best_sweep_seed200_v2_bestCSI.h5'},
    450: {'best_val_loss': 'results/best_sweep_seed450_v2.h5',
          'best_CSI':      'results/best_sweep_seed450_v2_bestCSI.h5'},
    800: {'best_val_loss': 'results/best_sweep_seed800_v2.h5',
          'best_CSI':      'results/best_sweep_seed800_v2_bestCSI.h5'},
}
# NOTE: seeds 1/200/450/800 are the _v2 reruns. The first versions (Jul 8-10) are
# invalid: concurrent jobs shared lightning_logs/finetune_ahr and overwrote each
# other's best.ckpt/best_csi.ckpt (fixed in run_finetune_hal8.sh with per-job
# --checkpoint-dir). Seed 666 ran alone, so it is clean.

# quick check which files are there
for seed, ckpts in CHECKPOINTS.items():
    for label, path in ckpts.items():
        print(f"seed {seed:3d} {label:14s} {'OK' if os.path.exists(path) else 'MISSING'}  {path}")

seed 666 best_val_loss  OK  results/best_sweep_new_gt.h5
seed 666 best_CSI       OK  results/best_sweep_new_gt_bestCSI.h5
seed   1 best_val_loss  OK  results/best_sweep_seed1_v2.h5
seed   1 best_CSI       OK  results/best_sweep_seed1_v2_bestCSI.h5
seed 200 best_val_loss  OK  results/best_sweep_seed200_v2.h5
seed 200 best_CSI       OK  results/best_sweep_seed200_v2_bestCSI.h5
seed 450 best_val_loss  OK  results/best_sweep_seed450_v2.h5
seed 450 best_CSI       OK  results/best_sweep_seed450_v2_bestCSI.h5
seed 800 best_val_loss  OK  results/best_sweep_seed800_v2.h5
seed 800 best_CSI       OK  results/best_sweep_seed800_v2_bestCSI.h5


## Evaluation function (same logic as run_inference.py)

In [4]:
def evaluate_checkpoint(ckpt_path):
    '''Load one checkpoint, run the full rollout, return epoch + metrics.'''
    model_parameters = dict(config.models)
    model_type = model_parameters.pop('model_type')
    if model_type == 'MSGNN':
        model_parameters['num_scales'] = test_dataset[0].mesh.num_meshes

    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    model = get_model(model_type)(
        num_node_features=num_node_features,
        num_edge_features=num_edge_features,
        previous_t=temporal_test_dataset_parameters['previous_t'],
        device=device,
        **model_parameters
    ).to(device)

    plmodule = LightningTrainer.load_from_checkpoint(
        ckpt_path, map_location=device,
        model=model,
        lr_info=config['lr_info'],
        trainer_options=config.trainer_options,
        temporal_test_dataset_parameters=temporal_test_dataset_parameters
    )
    model = plmodule.model.to(device)
    model.eval()

    plot_rollout = PlotRollout(
        model, test_dataset[0], scalers=scalers,
        warmup_steps=0,
        **temporal_test_dataset_parameters
    )
    rollout_loss = plot_rollout._get_rollout_loss(type_loss='RMSE')
    loss_mean = rollout_loss.mean(0)
    rmse_wd = loss_mean[0].item() if loss_mean.dim() > 0 else loss_mean.item()
    csi_005 = plot_rollout._get_CSI(water_threshold=0.05).nanmean().item()
    csi_03  = plot_rollout._get_CSI(water_threshold=0.30).nanmean().item()

    return ckpt.get('epoch', -1), rmse_wd, csi_005, csi_03

## Evaluate all seeds

(each full rollout takes a few minutes on CPU, so ~10-20 min for 10 checkpoints)

In [5]:
rows = []
for seed, ckpts in CHECKPOINTS.items():
    for label, path in ckpts.items():
        if not os.path.exists(path):
            print(f'skipped (missing): seed {seed} {label}')
            continue
        print(f'evaluating seed {seed} ({label})...')
        epoch, rmse_wd, csi005, csi03 = evaluate_checkpoint(path)
        rows.append(dict(seed=seed, ckpt=label, epoch=epoch,
                         RMSE_WD=rmse_wd, CSI_005=csi005, CSI_03=csi03))
        print(f'  epoch {epoch}: RMSE_WD={rmse_wd:.4f} m, CSI@0.05={csi005:.4f}, CSI@0.30={csi03:.4f}')

df = pd.DataFrame(rows)
df

evaluating seed 666 (best_val_loss)...


/u/marrocol/.conda/envs/mswe-gnn/lib/python3.10/site-packages/lightning/fabric/utilities/cloud_io.py:55: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


  epoch 922: RMSE_WD=0.1461 m, CSI@0.05=0.6800, CSI@0.30=0.8336
evaluating seed 666 (best_CSI)...


  epoch 632: RMSE_WD=0.1600 m, CSI@0.05=0.7409, CSI@0.30=0.7940
evaluating seed 1 (best_val_loss)...


  epoch 546: RMSE_WD=0.1539 m, CSI@0.05=0.6663, CSI@0.30=0.8114
evaluating seed 1 (best_CSI)...


...

  epoch 668: RMSE_WD=0.1459 m, CSI@0.05=0.7133, CSI@0.30=0.8250
evaluating seed 450 (best_CSI)...


  epoch 371: RMSE_WD=0.1543 m, CSI@0.05=0.7431, CSI@0.30=0.7895
evaluating seed 800 (best_val_loss)...


  epoch 660: RMSE_WD=0.1274 m, CSI@0.05=0.7297, CSI@0.30=0.8566
evaluating seed 800 (best_CSI)...


  epoch 347: RMSE_WD=0.1514 m, CSI@0.05=0.7977, CSI@0.30=0.8207


,seed,ckpt,epoch,RMSE_WD,CSI_005,CSI_03
0,666,best_val_loss,922,0.146111,0.679998,0.833597
1,666,best_CSI,632,0.159993,0.740881,0.794011
2,1,best_val_loss,546,0.153892,0.666292,0.811432
3,1,best_CSI,95,0.204289,0.711597,0.759685
4,200,best_val_loss,541,0.137438,0.726938,0.833874
5,200,best_CSI,287,0.174757,0.811095,0.694574
6,450,best_val_loss,668,0.145922,0.713319,0.824957
7,450,best_CSI,371,0.154308,0.743069,0.789459
8,800,best_val_loss,660,0.127405,0.729700,0.856552
9,800,best_CSI,347,0.151429,0.797687,0.820738


## The ruler: mean ± spread per checkpoint type

The `best_CSI` block is the headline (judge metric = CSI@0.05); `best_val_loss` is supporting.

The best epochs are also plateau evidence for Linea 1 (where training stops improving).

In [6]:
for label in ['best_CSI', 'best_val_loss']:
    sub = df[df.ckpt == label]
    if len(sub) < 2:
        continue
    print(f'=== RULER ({label}, n={len(sub)}) ===')
    for metric in ['RMSE_WD', 'CSI_005', 'CSI_03']:
        vals = sub[metric].values
        print(f'  {metric:8s}: mean={vals.mean():.4f} +- {vals.std(ddof=1):.4f}   range [{vals.min():.4f}, {vals.max():.4f}]')
    print(f'  best epochs: {sub.epoch.tolist()}')
    print()

=== RULER (best_CSI, n=5) ===
  RMSE_WD : mean=0.1690 +- 0.0217   range [0.1514, 0.2043]
  CSI_005 : mean=0.7609 +- 0.0419   range [0.7116, 0.8111]
  CSI_03  : mean=0.7717 +- 0.0482   range [0.6946, 0.8207]
  best epochs: [632, 95, 287, 371, 347]

=== RULER (best_val_loss, n=5) ===
  RMSE_WD : mean=0.1422 +- 0.0101   range [0.1274, 0.1539]
  CSI_005 : mean=0.7032 +- 0.0286   range [0.6663, 0.7297]
  CSI_03  : mean=0.8321 +- 0.0164   range [0.8114, 0.8566]
  best epochs: [922, 546, 541, 668, 660]



In [7]:
# save for the thesis appendix
df.to_csv('results/seed_table.csv', index=False)
print('saved results/seed_table.csv')

print(df.to_markdown(index=False, floatfmt='.4f'))

saved results/seed_table.csv
|   seed | ckpt          |   epoch |   RMSE_WD |   CSI_005 |   CSI_03 |
|-------:|:--------------|--------:|----------:|----------:|---------:|
|    666 | best_val_loss |     922 |    0.1461 |    0.6800 |   0.8336 |
|    666 | best_CSI      |     632 |    0.1600 |    0.7409 |   0.7940 |
|      1 | best_val_loss |     546 |    0.1539 |    0.6663 |   0.8114 |
|      1 | best_CSI      |      95 |    0.2043 |    0.7116 |   0.7597 |
|    200 | best_val_loss |     541 |    0.1374 |    0.7269 |   0.8339 |
|    200 | best_CSI      |     287 |    0.1748 |    0.8111 |   0.6946 |
|    450 | best_val_loss |     668 |    0.1459 |    0.7133 |   0.8250 |
|    450 | best_CSI      |     371 |    0.1543 |    0.7431 |   0.7895 |
|    800 | best_val_loss |     660 |    0.1274 |    0.7297 |   0.8566 |
|    800 | best_CSI      |     347 |    0.1514 |    0.7977 |   0.8207 |
